# Phenotype query + filtering check

Smoke test for the querying/filtering half of `02_phenotype`, run against the
real CDR before trusting the full `02_residualize_phenotypes.ipynb` pipeline.
Uses a short, high-confidence subset of `docs/phenotype_list.tsv` (the
confirmed anthropometric concept_ids only, no `UNCONFIRMED` lipid rows) to
check:

1. BigQuery connectivity under Workbench 2.0
2. the phenotype concept_ids resolve to the expected concepts
3. the measurement pull shape (rows, distinct persons, NA rate)
4. the round-2 keep-list intersection funnel
5. sex_at_birth / age join
6. implausible values (`filter_plausible_range()`) get dropped and the remainder is physiologically plausible
7. the real `run_residualization()` runs cleanly on real data, across all 4 covariate sets x {raw, invnorm}
8. covariate loadings, encoding, and phenotype distributions look sane (`base_pcs_zip3_ses` coefficients, SD/mean of PCs, zip3 factor levels, skewness)

**Every cell below prints aggregate/summary statistics only — counts, rates,
ranges — never a person-level row.** That's deliberate, not just for output
hygiene: it keeps this notebook safe to leave un-cleared if the outputs are
ever glanced at, on top of (not instead of) the repo-wide rule to clear
outputs / run `nbstripout` before committing anything run against real AoU
data (see root `README.md`).

## Compute resource

This notebook only issues small aggregate BigQuery queries and pulls a
handful of summary numbers back locally — BigQuery does the heavy lifting
server-side. Workbench 2.0's default cloud environment app size (2 CPU / 13
GB) is enough; no need to size up for this notebook. Reach for a larger
machine only for `02_residualize_phenotypes.ipynb` (holds the full cohort in
memory) or the GRM stages.

In [ ]:
required_pkgs <- c("dplyr", "readr", "stringr", "bigrquery", "allofus")
missing_pkgs <- required_pkgs[!sapply(required_pkgs, requireNamespace, quietly = TRUE)]
if (length(missing_pkgs) > 0) install.packages(missing_pkgs)

library(dplyr)
library(readr)
library(stringr)
library(bigrquery)
library(allofus)
source("../../scripts/local/residualize_lib.R")   # residualize_phenotype(), shared with 02_residualize_phenotypes.ipynb


## Connecting to the CDR

`allofus::aou_connect()` / `aou_sql()` — confirmed working on Workbench 2.0
in practice, same connection pattern `02_residualize_phenotypes.ipynb` uses.
`aou_sql()` interprets `{CDR}` in the query string via `glue`, resolved to
the workspace's CDR BigQuery dataset automatically — no manual
project/dataset/billing wiring needed.

In [ ]:
con <- aou_connect()   # BigQuery connection to the CDR

run_query <- function(sql) collect(aou_sql(sql))   # aou_sql() returns a lazy tbl; collect() materializes it

## Inputs

- Keep-list: `01_ancestry_pca_filter.ipynb`'s output (`01_ancestry_filtering`,
  two-round 1000G-referenced classification) for `SAMPLE_SET` below.
- PC covariates: `03_final_pca.ipynb`'s output (`PC_COVARIATE_PATH`) --
  PC1-PC10 fit within this exact `SAMPLE_SET`'s own final members, used in
  Step 5's `base_pcs` covariate set.
- Phenotype subset: the confirmed rows of `docs/phenotype_list.tsv`, first 3
  — enough to check the query/filter logic without pulling the full panel.
- Plausible-range bounds: `docs/phenotype_list.tsv`'s `plausible_min`/
  `plausible_max` columns, applied in Step 3 via `filter_plausible_range()`.

In [ ]:
WORKSPACE_BUCKET <- path.expand("~/workspace/Data from All of Us Controlled Tier /shared-env-pilot")

# Must match whatever CDR_VERSION 01_ancestry_filtering's notebooks were
# actually run with -- this notebook reads their output.
CDR_VERSION <- "v9"

# Top-level bucket folder name for this project's outputs -- distinct from
# CDR_VERSION, which keeps its real meaning elsewhere. Fixed literal, matches
# every other notebook in this pipeline.
PROJECT_DIR <- "phenotypic_covariance_v9"

ANCESTRY_BUCKET_DIR <- file.path(WORKSPACE_BUCKET, PROJECT_DIR, "01_ancestry_filtering")

# Which of 03_final_pca.ipynb's 6 sample sets to smoke-test against.
# prob_tag must match 01_ancestry_pca_filter.ipynb's own SAMPLE_SETS dict.
SAMPLE_SET <- "eur_base"
sample_set_cfg <- switch(SAMPLE_SET,
  eur_strict =  list(prob_tag = "p10"),
  eur_base =    list(prob_tag = "p50"),
  eur_loose =   list(prob_tag = "p99"),
  uniform = list(prob_tag = "uniform"),
  afr =         list(prob_tag = "p2"),
  eas =         list(prob_tag = "p90")
)

# no variant panel now, classification is over AoU premade PCs -- 01_ancestry_pca_filter.ipynb
# writes keep-lists directly under ancestry_panel/final_pca/
FINAL_PCA_DIR <- file.path(ANCESTRY_BUCKET_DIR, "ancestry_pca_filter", "final_pca")

KEEP_LIST_PATH <- file.path(FINAL_PCA_DIR, paste0("final_keep_ids_", SAMPLE_SET, "_", sample_set_cfg$prob_tag, ".txt"))   # 01_ancestry_pca_filter.ipynb output
PC_COVARIATE_PATH <- file.path(FINAL_PCA_DIR, SAMPLE_SET, paste0("final_pca_pc_covariates_", SAMPLE_SET, ".txt"))   # 03_final_pca.ipynb output -- PC1-PC10 fit within this exact sample set

read_keep_list <- function(path) {
  # plink-style: one ID per line, or "FID IID" space-separated -- take the
  # last whitespace-separated field either way
  str_trim(read_lines(path)) %>% str_split(" +") %>% sapply(function(x) tail(x, 1))
}

keep_ids <- read_keep_list(KEEP_LIST_PATH)

tibble(
  keep_list = c(paste0("final/", SAMPLE_SET)),
  n_ids = c(length(keep_ids))
)

In [ ]:
pheno_list <- read_tsv("../../docs/phenotype_list.tsv", col_types = cols(.default = "c")) %>%
  filter(concept_id != "UNCONFIRMED") %>%
  slice_head(n = 3)

pheno_list

## Step 1 — concept sanity check

`{CDR}.concept` is public OMOP vocabulary metadata, not participant data —
safe to print in full. Confirms each concept_id exists, is in the expected
domain, and is a standard concept before spending a bigger query on it.

In [ ]:
concept_ids <- paste(pheno_list$concept_id, collapse = ",")
concept_check <- run_query(sprintf("
  SELECT concept_id, concept_name, domain_id, vocabulary_id, standard_concept
  FROM {CDR}.concept
  WHERE concept_id IN (%s)
", concept_ids))

concept_check

## Step 2 — measurement pull funnel

For each phenotype: total measurement rows for its concept_id, distinct
persons, and the NA rate on `value_as_number` — all aggregate counts, no
person-level output.

In [ ]:
pull_funnel_row <- function(pheno_row) {
  stats <- run_query(sprintf("
    SELECT
      COUNT(*) AS n_rows,
      COUNT(DISTINCT person_id) AS n_persons,
      COUNTIF(value_as_number IS NULL) AS n_null_value
    FROM {CDR}.measurement
    WHERE measurement_concept_id = %s
  ", pheno_row$concept_id))

  stats %>% mutate(phenotype_name = pheno_row$phenotype_name, .before = 1)
}

pull_funnel <- bind_rows(lapply(seq_len(nrow(pheno_list)), function(i) pull_funnel_row(pheno_list[i, ])))
pull_funnel

## Step 3 — plausible-range + keep-list filtering, sex/age join

Most-recent-value-per-person, joined to age and `sex_at_birth_concept_id`
(the AoU-specific column, distinct from `gender_concept_id`) — same shape
`pull_phenotype()` returns in `02_residualize_phenotypes.ipynb`. Applies
`filter_plausible_range()` (bounds from `docs/phenotype_list.tsv`) before
the keep-list filter, same order `run_residualization()` uses, so it's
shared by every downstream covariate-set combo. Reports the funnel from raw
pull through the range filter to the keep-list intersection, plus the
sex_at_birth category breakdown (aggregate counts only).

In [ ]:
REFERENCE_DATE <- "2024-01-01"   # arbitrary fixed date -- adjust to match the CDR version's actual data cutoff
stopifnot(REFERENCE_DATE != "PLACEHOLDER")   # unset fails as an invalid DATE literal deep inside the BigQuery
                                              # job, surfaced by aou_sql() as an opaque "did not result in a table" error

pull_and_filter <- function(pheno_row) {
  # DATE_DIFF returns INT64; bigrquery collects bare INT64 columns as bit64::integer64,
  # which lm() silently mis-coerces into a degenerate fit rather than erroring -- cast to
  # FLOAT64 in the query so age collects as a plain double
  query <- sprintf("
    WITH demographics AS (
      SELECT
        person_id,
        CAST(DATE_DIFF(DATE '%s', DATE(birth_datetime), YEAR) AS FLOAT64) AS age,
        CASE
          WHEN sex_at_birth_concept_id = 45878463 THEN 'Female'
          WHEN sex_at_birth_concept_id = 45880669 THEN 'Male'
          ELSE 'Other'
        END AS sex_at_birth
      FROM {CDR}.person
    ),
    measurements AS (
      SELECT
        person_id,
        value_as_number AS phenotype,
        ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY measurement_date DESC) AS rn
      FROM {CDR}.measurement
      WHERE measurement_concept_id = %s
        AND value_as_number IS NOT NULL
    )
    SELECT d.person_id, m.phenotype, d.age, d.sex_at_birth
    FROM demographics d
    INNER JOIN measurements m ON d.person_id = m.person_id
    WHERE m.rn = 1
  ", REFERENCE_DATE, pheno_row$concept_id)

  df <- run_query(query) %>%
    mutate(person_id = as.character(person_id))

  # physiologically-plausible range filter (filter_plausible_range() in residualize_lib.R,
  # bounds from docs/phenotype_list.tsv) -- applied before the keep-list filter so it's
  # shared by every downstream covariate-set combo, matching run_residualization()'s order
  range_result <- filter_plausible_range(
    df, "phenotype", as.numeric(pheno_row$plausible_min), as.numeric(pheno_row$plausible_max)
  )
  df <- range_result$data

  df_filtered <- df %>% filter(person_id %in% keep_ids)

  list(name = pheno_row$phenotype_name, df = df, df_filtered = df_filtered,
       n_raw = range_result$n_input, n_excluded_implausible = range_result$n_excluded)
}

pulls <- lapply(seq_len(nrow(pheno_list)), function(i) pull_and_filter(pheno_list[i, ]))
names(pulls) <- pheno_list$phenotype_name

filter_funnel <- bind_rows(lapply(pulls, function(p) {
  tibble(
    phenotype_name = p$name,
    n_raw = p$n_raw,
    n_excluded_implausible = p$n_excluded_implausible,
    n_after_range_filter = nrow(p$df),
    n_after_keep_list = nrow(p$df_filtered)
  )
}))
filter_funnel

In [ ]:
sex_breakdown <- bind_rows(lapply(pulls, function(p) {
  p$df_filtered %>% count(sex_at_birth) %>% mutate(phenotype_name = p$name, .before = 1)
}))
sex_breakdown

## Step 4 — value range sanity check

Aggregate summary stats per phenotype (min/median/mean/max/SD, NA rate) on
the keep-list-filtered values — already past `filter_plausible_range()`
from Step 3, so these ranges should look physiologically plausible by
construction; this is really a check that the bounds in
`docs/phenotype_list.tsv` weren't set wrong, not a fresh discovery step.

In [ ]:
value_summary <- bind_rows(lapply(pulls, function(p) {
  x <- p$df_filtered$phenotype
  tibble(
    phenotype_name = p$name,
    n = sum(!is.na(x)),
    min = min(x, na.rm = TRUE),
    median = median(x, na.rm = TRUE),
    mean = mean(x, na.rm = TRUE),
    max = max(x, na.rm = TRUE),
    sd = sd(x, na.rm = TRUE)
  )
}))
value_summary

## Step 5 — the 4 residualization models, raw + invnorm

Runs `residualize_lib.R`'s real `run_residualization()` — the convenience
wrapper around the same `prepare_modeling_tables()` /
`run_residualization_from_tables()` two-stage pipeline
`02_residualize_phenotypes.ipynb` calls explicitly (there's no bucket table
worth persisting for a 3-phenotype smoke test, so the wrapper's default
throwaway `tempfile()` table dir is fine here) — on the phenotypes already
pulled in Step 3, using `build_covariate_sets()`'s nested staircase:

1. `base` — age only
2. `base_pcs` — + PC1..PC5
3. `base_pcs_zip3` — + 3-digit zip (factor)
4. `base_pcs_zip3_ses` — + median_income/poverty/deprivation_index

Each of those also gets crossed with `{raw, invnorm}` automatically —
`prepare_modeling_tables()` calls `add_transformed_variant()` internally to
build the rank-based inverse-normal-transformed version, and
`run_residualization_from_tables()` residualizes both, so this is also the
"unskewed version, regress covariates out of it too" check.
`pull_phenotype_prepulled()`/`pull_covariates_prepulled()` below just wrap
the data already pulled in Steps 3 and below — no new BigQuery calls — so
this exercises the real pipeline logic on real AoU values, not synthetic
data like `test_residualize_fake_data.ipynb`. `.pheno`-shaped files get
written to a local `/tmp` scratch directory, same convention as that
fake-data test — throwaway, not the workspace bucket.

In [ ]:
pcs <- read_table(PC_COVARIATE_PATH, col_types = cols(.default = "d", IID = "c")) %>%
  rename(person_id = IID)
pc_cols <- paste0("PC", 1:5)   # top 5 only -- beyond that isn't considered informative for this cohort

### Pull SES data

Same `zip3_ses_map` join `pull_covariates()` uses in
`02_residualize_phenotypes.ipynb`, with the same `FLOAT64` casts from the
integer64 fix above. Not phenotype-specific, so pulled once.

In [ ]:
ses_query <- "
  SELECT
    o.person_id,
    o.zip3,
    CAST(z.median_income AS FLOAT64) AS median_income,
    CAST(z.fraction_poverty AS FLOAT64) AS poverty,
    CAST(z.deprivation_index AS FLOAT64) AS deprivation_index
  FROM (
    SELECT person_id, CAST(SUBSTR(value_as_string, 1, 3) AS INT64) AS zip3
    FROM {CDR}.observation
    WHERE STRPOS(value_as_string, '*') > 0
  ) o
  INNER JOIN {CDR}.zip3_ses_map z ON o.zip3 = z.zip3
"

ses <- run_query(ses_query) %>%
  mutate(person_id = as.character(person_id), zip3 = as.character(zip3))   # zip3 is a factor
                                                                            # level label, not numeric -- cast
                                                                            # in R same as pull_covariates()
nrow(ses)

In [ ]:
covariate_sets <- build_covariate_sets(pc_cols)

pull_phenotype_prepulled <- function(row, keep_ids) {
  pulls[[row$phenotype_name]]$df_filtered %>% filter(person_id %in% keep_ids)
}

pull_covariates_prepulled <- function(keep_ids) {
  list(
    pcs = pcs %>% filter(person_id %in% keep_ids),
    zip3 = ses %>% transmute(person_id, zip3) %>% filter(person_id %in% keep_ids),
    ses = ses %>% select(person_id, median_income, poverty, deprivation_index) %>% filter(person_id %in% keep_ids)
  )
}

MOCK_OUT_DIR <- "/tmp/query_filter_check_mock_out"
unlink(MOCK_OUT_DIR, recursive = TRUE)

mock_result <- run_residualization(
  pheno_list, keep_ids, pull_phenotype_prepulled, pull_covariates_prepulled,
  covariate_sets, MOCK_OUT_DIR
)

# will show 0 additional exclusions -- Step 3 already applied filter_plausible_range();
# shown here for parity with what the real pipeline's diagnostics look like
mock_result$range_summary_table

In [ ]:
mock_result$skew_summary_table   # raw vs invnorm skewness per phenotype

In [ ]:
mock_result$combo_summary_table  # all 4 covariate sets x {raw, invnorm} -- N retained, R^2 per model

## Step 6 — model diagnostics: loadings, encoding, distributions

Deeper look than Step 5's retained-N/R² summary: coefficient tables (how
each phenotype loads onto age, each PC, zip3, and the SES variables), how
covariates actually get encoded going into `lm()`, and the raw/transformed
phenotype distributions. Everything below is model-level output
(coefficients, encoding metadata, aggregate distributions/quantiles) —
never a person-level row.

### Fit `base_pcs_zip3_ses` (the full model) and pull out coefficients

Fits the richest of the 4 covariate sets directly with `lm()` (not
`residualize_phenotype()`, which returns diagnostics but not the fit
object) so the coefficients themselves are inspectable — one fit per
phenotype, reused below for both the age/PC/SES coefficient tables and the
zip3 factor-encoding check. Joins are `inner_join`, so this implicitly
restricts to people with both PCs (round 2 passers) and a masked
zip3/SES record.

In [ ]:
tidy_lm <- function(fit) {
  s <- summary(fit)$coefficients
  tibble(
    term = rownames(s),
    estimate = s[, "Estimate"],
    std_error = s[, "Std. Error"],
    t_value = s[, "t value"],
    p_value = s[, "Pr(>|t|)"]
  )
}

full_fits <- lapply(pulls, function(p) {
  df <- p$df %>% inner_join(pcs, by = "person_id") %>% inner_join(ses, by = "person_id")
  formula <- as.formula(paste("phenotype ~", paste(covariate_sets$base_pcs_zip3_ses, collapse = " + ")))
  lm(formula, data = df, na.action = na.exclude)
})
names(full_fits) <- names(pulls)

sapply(full_fits, function(f) sum(!is.na(residuals(f))))   # N actually used per phenotype's fit

In [ ]:
coef_table <- bind_rows(lapply(names(full_fits), function(name) {
  tidy_lm(full_fits[[name]]) %>% mutate(phenotype_name = name, .before = 1)
}))

# age + SES loadings -- the PC loadings are in the next cell, kept separate since
# there are 5 of them, and zip3's dummy terms are summarized further below
coef_table %>% filter(term %in% c("age", "median_income", "poverty", "deprivation_index"))

In [ ]:
coef_table %>% filter(grepl("^PC", term))

### Covariate encoding check

Confirms `age`/PCs/SES collected as plain numeric doubles (the integer64
fix above) rather than `bit64::integer64`, and that plink2's
`variance-standardize` scoring actually left the PCs at roughly mean-0/
SD-1. `sex_at_birth` is **not** a regression covariate here — per
`residualize_lib.R`, it's the stratification variable for the separate
within-sex standardization step (step 3 in the module docstring), so
there's no `lm()` factor/contrast encoding for it to check; what's worth
confirming instead is that its values are the expected small category set.

In [ ]:
sapply(pulls[[1]]$df["age"], class)
sapply(pcs[pc_cols], class)
sapply(ses[c("median_income", "poverty", "deprivation_index")], class)

# variance-standardize scoring should leave these close to mean 0, SD 1
sapply(pcs[pc_cols], function(x) c(mean = mean(x, na.rm = TRUE), sd = sd(x, na.rm = TRUE)))

In [ ]:
sex_breakdown_all <- bind_rows(lapply(names(pulls), function(name) {
  pulls[[name]]$df_filtered %>% count(sex_at_birth) %>% mutate(phenotype_name = name, .before = 1)
}))
sex_breakdown_all

### zip3 as a factor covariate

`build_covariate_sets()`'s `base_pcs_zip3`/`base_pcs_zip3_ses` steps use
the 3-digit zip as a categorical covariate. `lm()` one-hot-encodes a
character column automatically (treatment contrasts, reference level =
alphabetically first), so `base_pcs_zip3_ses` — already fit above as
`full_fits` — produces one dummy term per zip3 level minus one. With ~800
possible 3-digit US zip codes, printing every dummy's coefficient by
default would be a lot of noise — summarized below instead; `full_fits`
has the full `lm` objects if you want to inspect specific levels.

In [ ]:
# aggregate zip3 frequency -- geographic distribution, not person-level
ses %>% count(zip3, sort = TRUE) %>% head(10)

In [ ]:
zip3_fit_summary <- bind_rows(lapply(names(full_fits), function(name) {
  fit <- full_fits[[name]]
  s <- summary(fit)$coefficients
  zip3_rows <- grepl("^zip3", rownames(s))
  tibble(
    phenotype_name = name,
    n_zip3_levels_in_data = length(unique(fit$model$zip3)),
    n_zip3_dummy_terms = sum(zip3_rows),
    n_zip3_dummies_sig_p05 = sum(zip3_rows & s[, "Pr(>|t|)"] < 0.05, na.rm = TRUE),
    r_squared = summary(fit)$r.squared,
    n_used = nrow(fit$model)
  )
}))
zip3_fit_summary

### Phenotype distributions

Histograms (raw), a skewness table (raw vs. `inverse_normal_transform()`,
reusing `residualize_lib.R`'s real functions rather than reimplementing),
quantiles, and a by-sex boxplot — all aggregate/visual, no person-level
values printed.

In [ ]:
for (name in names(pulls)) {
  hist(pulls[[name]]$df_filtered$phenotype, breaks = 40, main = paste(name, "-- raw"), xlab = name)
}

In [ ]:
skew_table <- bind_rows(lapply(names(pulls), function(name) {
  x <- pulls[[name]]$df_filtered$phenotype
  bind_rows(
    skew_summary(x, "raw"),
    skew_summary(inverse_normal_transform(x), "invnorm")
  ) %>% mutate(phenotype = name, .before = 1)
}))
skew_table

In [ ]:
quantile_table <- bind_rows(lapply(names(pulls), function(name) {
  x <- pulls[[name]]$df_filtered$phenotype
  q <- quantile(x, probs = c(0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99), na.rm = TRUE)
  tibble(phenotype_name = name, quantile = names(q), value = as.numeric(q))
}))
quantile_table

In [ ]:
for (name in names(pulls)) {
  df <- pulls[[name]]$df_filtered
  boxplot(phenotype ~ sex_at_birth, data = df, main = paste(name, "by sex_at_birth"), ylab = name)
}

## Summary

If `concept_check` matched the expected concepts, `filter_funnel` shows a
sane retention rate (and a small, non-dominant `n_excluded_implausible`
relative to `n_raw` — if it's a large fraction, the bounds in
`docs/phenotype_list.tsv` are probably wrong, not the data), `value_summary`'s
ranges look physiologically plausible, `mock_result$combo_summary_table`
shows plausible retained-N/R² across all 4 covariate sets x {raw, invnorm},
and Step 6's `coef_table`/`zip3_fit_summary`/encoding checks/distributions
all look sane — the query, filtering, and residualization logic is
validated and `02_residualize_phenotypes.ipynb` is safe to run on the full
phenotype list.

Still needs the real workbench before that: confirming `WORKSPACE_BUCKET`
matches the actual mounted resource name, and that `REFERENCE_DATE` is
appropriate for the CDR version in use.

## Appendix — searching for a better waist_circumference/hip_circumference concept

`waist_circumference`'s confirmed concept (LOINC 8280-0, concept_id 3016258)
has only 22 persons in the whole CDR (see `docs/phenotype_list.tsv`'s
notes) — essentially unusable, and the note there explicitly says it's
worth revisiting "unless a materially different source for waist
circumference turns up." Checks `MODELING_TABLE_DIR` (same bucket path
`02_residualize_phenotypes.ipynb` writes to) first — if `waist_circumference.tsv`/
`hip_circumference.tsv` are already there from a prior run, skips straight
to reporting their row counts rather than re-querying. Otherwise searches
`{CDR}.concept` across *all* vocabularies (not just LOINC) for any
measurement concept whose name mentions waist/hip circumference, joined to
`{CDR}.measurement` for a real person count per candidate — the same
concept-then-count pattern Step 1/Step 2 use above, just applied to a
broader name search instead of a fixed concept_id list. Every row printed
is aggregate concept metadata + counts, never person-level.

In [ ]:
PHENOTYPE_BUCKET_DIR <- file.path(WORKSPACE_BUCKET, PROJECT_DIR, "02_phenotype")
MODELING_TABLE_DIR <- file.path(PHENOTYPE_BUCKET_DIR, "modeling_tables")   # same path 02_residualize_phenotypes.ipynb writes to

waist_table_path <- file.path(MODELING_TABLE_DIR, "waist_circumference.tsv")
hip_table_path <- file.path(MODELING_TABLE_DIR, "hip_circumference.tsv")

if (file.exists(waist_table_path) && file.exists(hip_table_path)) {
  message("waist_circumference.tsv and hip_circumference.tsv already exist in MODELING_TABLE_DIR -- skipping concept search")
  bind_rows(
    tibble(phenotype_name = "waist_circumference", n = nrow(read_tsv(waist_table_path, show_col_types = FALSE))),
    tibble(phenotype_name = "hip_circumference", n = nrow(read_tsv(hip_table_path, show_col_types = FALSE)))
  )
} else {
  message("waist_circumference.tsv/hip_circumference.tsv not found in MODELING_TABLE_DIR -- searching {CDR}.concept for alternative concepts")

  candidate_concepts <- run_query("
    SELECT concept_id, concept_name, domain_id, vocabulary_id, standard_concept
    FROM {CDR}.concept
    WHERE domain_id = 'Measurement'
      AND (LOWER(concept_name) LIKE '%waist circumference%' OR LOWER(concept_name) LIKE '%hip circumference%')
  ")

  candidate_counts <- bind_rows(lapply(candidate_concepts$concept_id, function(cid) {
    run_query(sprintf("
      SELECT COUNT(DISTINCT person_id) AS n_persons, COUNT(*) AS n_rows
      FROM {CDR}.measurement
      WHERE measurement_concept_id = %s
    ", cid)) %>% mutate(concept_id = cid, .before = 1)
  }))

  candidate_concepts %>%
    inner_join(candidate_counts, by = "concept_id") %>%
    arrange(desc(n_persons))
}

## Appendix — investigating hemoglobin's high implausible-range exclusion rate

`hemoglobin` (LOINC 718-7, concept_id 3000963, n=280,701 persons confirmed)
has had roughly 100,000 measurements (~36%) dropped by
`filter_plausible_range()`'s current `3`-`20` g/dL bound in a real run of
`02_residualize_phenotypes.ipynb`. That's far too high a rate for a
deliberately generous physiological bound — real hemoglobin essentially
never falls outside 3-20 g/dL if it's actually measured in g/dL. The
likelier explanation is a **unit mismatch**: LOINC 718-7 aggregates
hemoglobin results from many different source EHR systems, and some may
report in g/L (SI convention — normal is ~120-175 g/L) rather than g/dL,
which would blow straight past the upper bound.

**Resolved below:** on the full unrestricted cohort the actual exclusion
rate was even worse (90.5%) than the ~36% first reported, and the
free-text `unit_source_value` label turned out unreliable even for the
*dominant* group (labeled `"g/dL"`, but really g/L) — the fix that
actually works keys off `unit_concept_id` instead, dropping the exclusion
rate to 0.8%. See the "Constructing a unit-normalized hemoglobin value"
cell below for the full story, including the first (wrong) attempt.

This cell doesn't touch `plausible_min`/`plausible_max` — it groups
hemoglobin's raw `value_as_number` by `unit_concept_id`/`unit_source_value`
and reports the value distribution and the out-of-3-20 count *within each
unit group*, as the starting point for that investigation. All output is
aggregate (grouped counts/stats), never person-level.

In [ ]:
HEMOGLOBIN_CONCEPT_ID <- 3000963
HEMOGLOBIN_PLAUSIBLE_MIN <- 3
HEMOGLOBIN_PLAUSIBLE_MAX <- 20

# allofus-idiomatic version: aou_concept_set() (the package's high-level helper) drops
# unit_source_value entirely (confirmed from its source -- it only keeps unit_concept_id),
# which is exactly the column that separates "g/dL" (real) from "g/dL{calc}"/raw SNOMED
# codes (wrong scale) below. So this uses the package's documented fallback for anything
# not covered by the high-level helpers: dplyr verbs directly against tbl(con, "measurement")
# (con from aou_connect() above), not a hand-written SQL string via aou_sql().
hemoglobin_by_unit <- tbl(con, "measurement") %>%
  filter(measurement_concept_id == HEMOGLOBIN_CONCEPT_ID, !is.na(value_as_number)) %>%
  left_join(tbl(con, "concept"), by = c("unit_concept_id" = "concept_id")) %>%
  group_by(unit_concept_id, unit_name = concept_name, unit_source_value) %>%
  summarise(
    n_rows = n(),
    n_persons = n_distinct(person_id),
    min_value = min(value_as_number, na.rm = TRUE),
    median_value = median(value_as_number, na.rm = TRUE),
    max_value = max(value_as_number, na.rm = TRUE),
    # BigQuery's SUM() rejects a BOOL argument directly (unlike R, no implicit BOOL->INT64
    # coercion) -- as.integer() first so this translates to SUM(CASE WHEN ... THEN 1 ELSE 0 END)
    n_outside_plausible_range = sum(as.integer(value_as_number < HEMOGLOBIN_PLAUSIBLE_MIN | value_as_number > HEMOGLOBIN_PLAUSIBLE_MAX), na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(n_rows)) %>%
  collect()

hemoglobin_by_unit

### Constructing a unit-normalized hemoglobin value

The fix isn't a wider `plausible_min`/`plausible_max` -- that would just let
mis-scaled values through as garbage instead of correcting them.

**First attempt (wrong):** grouping by `unit_source_value`'s free-text label
and converting only the two groups whose group-wide minimum sat above the
plausible ceiling (`"g/dL{calc}"`, the raw SNOMED code `"258795003"`) barely
moved the exclusion rate (90.5% -> 88.4% on the full unrestricted cohort).
That's because the free-text label lied for the *dominant* group too: a
direct check of `unit_source_value == "g/dL"` in isolation (261,417
persons -- the large majority of this concept's data) showed
`median_value = 124` -- clearly g/L, not g/dL, despite the label saying
"g/dL". The label can't be trusted at all.

**What actually works:** every sizeable group here (`"g/dL"` literal,
`"g/dL{calc}"`, the SNOMED-code one, `"g/L"` itself) shares the same
`unit_concept_id` (8636, "gram per liter") regardless of what the
free-text label claims -- and we've now directly verified that
concept_id is telling the truth. So the real rule keys off
`unit_concept_id`, not `unit_source_value`: convert everything mapped to
8636, leave `unit_concept_id` 0 ("No matching concept")/NA (genuinely
unknown unit) as reported. Confirmed against the real CDR: this drops the
exclusion rate from 90.5% to **0.8%** on the full unrestricted cohort --
the earlier `unit_source_value`-based version is superseded below.

Pulls the same most-recent-per-person shape `pull_and_filter()` uses,
carries unit info along, applies the `unit_concept_id`-keyed `/10`
conversion, then reruns `filter_plausible_range()` with the *same* 3-20
bound on both the original and normalized values side by side.

In [ ]:
# same most-recent-per-person shape as pull_and_filter() above, via allofus/dplyr against
# tbl(con, "measurement") rather than a raw SQL string -- dbplyr::window_order() (not plain
# arrange(), and not exported by dplyr itself -- needs the dbplyr:: prefix or library(dbplyr))
# is dbplyr's documented way to make row_number() deterministic inside a grouped filter,
# since SQL itself has no inherent row order
hemoglobin_most_recent <- tbl(con, "measurement") %>%
  filter(measurement_concept_id == HEMOGLOBIN_CONCEPT_ID, !is.na(value_as_number)) %>%
  group_by(person_id) %>%
  dbplyr::window_order(desc(measurement_date)) %>%
  filter(row_number() == 1) %>%
  ungroup() %>%
  select(person_id, value_as_number, unit_concept_id, unit_source_value) %>%
  collect()

# unit_concept_id (not the free-text unit_source_value label) is the reliable signal --
# verified directly: the dominant "g/dL"-labeled group's own median is 124 (i.e. truly
# g/L despite its label), and it shares unit_concept_id 8636 ("gram per liter") with
# every other sizeable group here regardless of what their source-text label says. So
# convert everything mapped to 8636; leave unit_concept_id 0 ("No matching concept")/NA
# (genuinely unknown unit) as reported -- no principled correction for those.
GRAM_PER_LITER_UNIT_CONCEPT_ID <- 8636

hemoglobin_normalized <- hemoglobin_most_recent %>%
  mutate(value_normalized = ifelse(unit_concept_id == GRAM_PER_LITER_UNIT_CONCEPT_ID,
                                    value_as_number / 10, value_as_number))

range_before <- filter_plausible_range(
  hemoglobin_most_recent %>% rename(phenotype = value_as_number),
  "phenotype", HEMOGLOBIN_PLAUSIBLE_MIN, HEMOGLOBIN_PLAUSIBLE_MAX
)
range_after <- filter_plausible_range(
  hemoglobin_normalized %>% rename(phenotype = value_normalized),
  "phenotype", HEMOGLOBIN_PLAUSIBLE_MIN, HEMOGLOBIN_PLAUSIBLE_MAX
)

# confirmed against the real CDR: 90.5% -> 0.8% excluded
tibble(
  stage = c("before unit normalization", "after unit normalization"),
  n_input = c(range_before$n_input, range_after$n_input),
  n_excluded = c(range_before$n_excluded, range_after$n_excluded),
  pct_excluded = round(100 * c(range_before$n_excluded, range_after$n_excluded) / range_before$n_input, 1)
)

## Appendix — checking concept_ids for lifestyle phenotypes (alcohol, tobacco)

`docs/candidate_phenotypes.tsv` has `alcohol_audit_c_score` (`source ==
"survey_composite"`, 3 AUDIT-C items summed) and `cigarettes_per_day`
(`source == "survey"`) staged, but both still have `UNCONFIRMED`
concept_id(s) -- these are survey/PPI answers, so they live in
`{CDR}.observation`, not `{CDR}.measurement`, and the AoU Data Browser's
"Survey" domain (not "Measurement") is the place to browse them
interactively. This cell does the same concept-then-count pattern as the
waist/hip appendix above, just pointed at `{CDR}.concept`'s PPI vocabulary
and name-searched for alcohol/drink and smoking/tobacco/cigarette keywords,
then counts real `{CDR}.observation` rows/persons per candidate so you can
tell which concept_id(s) are actually the live AUDIT-C items / tobacco item
in this CDR version, not just plausible-looking names. `concept_code` is
included since AoU's PPI codes (e.g. `Alcohol_*`, `Smoking_*`) are often a
clearer signal than the free-text `concept_name` alone. All output is
aggregate concept metadata + counts, never person-level.

In [ ]:
lifestyle_candidate_concepts <- run_query("
  SELECT concept_id, concept_name, concept_code, domain_id, vocabulary_id, standard_concept
  FROM {CDR}.concept
  WHERE vocabulary_id = 'PPI'
    AND (
      LOWER(concept_name) LIKE '%alcohol%' OR LOWER(concept_name) LIKE '%drink%'
      OR LOWER(concept_name) LIKE '%smoking%' OR LOWER(concept_name) LIKE '%tobacco%'
      OR LOWER(concept_name) LIKE '%cigarette%'
    )
")

lifestyle_candidate_counts <- bind_rows(lapply(lifestyle_candidate_concepts$concept_id, function(cid) {
  run_query(sprintf("
    SELECT COUNT(DISTINCT person_id) AS n_persons, COUNT(*) AS n_rows
    FROM {CDR}.observation
    WHERE observation_concept_id = %s
  ", cid)) %>% mutate(concept_id = cid, .before = 1)
}))

lifestyle_candidate_concepts %>%
  inner_join(lifestyle_candidate_counts, by = "concept_id") %>%
  arrange(desc(n_persons))

## Appendix — checking every concept_id in the full phenotype_list.tsv, one at a time

Unlike Step 1 above (a single `IN (...)` batched query over a 3-row smoke-test
subset), this reads the *entire* `docs/phenotype_list.tsv` and checks each
`concept_id` individually against `{CDR}.concept` -- a bad or non-numeric ID
in a batched `IN (...)` query fails the whole query with a SQL error, so this
isolates each check instead (`tryCatch`), reporting per-row status rather than
losing the good IDs' results to one bad one. Rows with multiple
comma-separated concept_ids (`survey_composite` items, `derived_ratio`'s
`"UNCONFIRMED,UNCONFIRMED"`) are split via base R `strsplit()` (not
`tidyr::separate_longer_delim()`, which needs tidyr >= 1.3.0 -- not in this
notebook's `required_pkgs` and not guaranteed to be the installed version) so
each ID gets its own check. `UNCONFIRMED`/non-numeric entries are skipped
without hitting BigQuery at all. All output is aggregate concept metadata,
never person-level.

In [ ]:
full_pheno_list <- read_tsv("../../docs/phenotype_list.tsv", col_types = cols(.default = "c"))

# Some rows hold multiple comma-separated concept_ids (survey_composite items,
# derived_ratio's "UNCONFIRMED,UNCONFIRMED") -- split so each individual ID
# gets its own row/check, not one glued-together string. Base R (strsplit),
# not tidyr::separate_longer_delim() -- that needs tidyr >= 1.3.0, which
# isn't in this notebook's required_pkgs and may not be the installed version.
split_ids <- strsplit(full_pheno_list$concept_id, ",")
full_concept_ids_flat <- tibble(
  phenotype_name = rep(full_pheno_list$phenotype_name, lengths(split_ids)),
  concept_id = str_trim(unlist(split_ids))
)

check_one_concept <- function(phenotype_name, concept_id) {
  # skip placeholders without hitting BigQuery at all -- querying
  # `WHERE concept_id = UNCONFIRMED` would just error anyway
  if (is.na(concept_id) || concept_id == "" || !grepl("^[0-9]+$", concept_id)) {
    return(tibble(
      phenotype_name, concept_id, status = "SKIPPED (not a numeric concept_id -- still UNCONFIRMED or malformed)",
      concept_name = NA_character_, domain_id = NA_character_,
      vocabulary_id = NA_character_, standard_concept = NA_character_
    ))
  }

  tryCatch({
    result <- run_query(sprintf("
      SELECT concept_id, concept_name, domain_id, vocabulary_id, standard_concept
      FROM {CDR}.concept
      WHERE concept_id = %s
    ", concept_id))

    if (nrow(result) == 0) {
      tibble(phenotype_name, concept_id, status = "NOT FOUND in {CDR}.concept",
             concept_name = NA_character_, domain_id = NA_character_,
             vocabulary_id = NA_character_, standard_concept = NA_character_)
    } else {
      result %>% mutate(phenotype_name, status = "ok", .before = 1) %>%
        mutate(concept_id = as.character(concept_id))
    }
  }, error = function(e) {
    tibble(phenotype_name, concept_id, status = paste("ERROR:", conditionMessage(e)),
           concept_name = NA_character_, domain_id = NA_character_,
           vocabulary_id = NA_character_, standard_concept = NA_character_)
  })
}

full_concept_check_results <- bind_rows(mapply(
  check_one_concept,
  full_concept_ids_flat$phenotype_name,
  full_concept_ids_flat$concept_id,
  SIMPLIFY = FALSE
))

# ok first, then problems, so anything needing attention floats to the top of the tail
full_concept_check_results %>% arrange(status != "ok", phenotype_name)